# The ETS Framework

ETS stands for **Error, Trend, Seasonal** — a state-space formulation that
provides a **unified taxonomy** for all exponential smoothing methods.  By
specifying the type of error (additive or multiplicative), trend (none,
additive, or additive-damped), and seasonal component (none, additive, or
multiplicative), we get a family of 18 models — of which 15 are numerically
stable and usable.

The ETS framework enables:
- **Automatic model selection** via information criteria (AIC, BIC).
- **Proper likelihood-based estimation** of parameters.
- **Prediction intervals** from the state-space formulation.

**Key references:**
- FPP3 Section 8.5–8.7
- Hyndman et al. (2002): *A State Space Framework for Automatic Forecasting*
- Hyndman et al. (2008): *Forecasting with Exponential Smoothing*

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

## 1. Load Data

In [ ]:
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"
airline.columns = ["Passengers"]

# Train/test split
train = airline.iloc[:120]
test = airline.iloc[120:]
HORIZON = len(test)

print(f"Train: {len(train)} months, Test: {len(test)} months")
airline.head()

---
## 2. The ETS Taxonomy

### Methods (ignoring error type)

When we only consider the **trend** and **seasonal** components, we get
the 9 methods that correspond to the exponential smoothing models from the
previous notebooks:

| | **N** (None) | **A** (Additive) | **M** (Multiplicative) |
|---|---|---|---|
| **N** (No trend) | (N,N) SES | (N,A) Seasonal, no trend | (N,M) Seasonal, no trend |
| **A** (Additive trend) | (A,N) Holt's linear | (A,A) Additive HW | (A,M) Multiplicative HW |
| **A_d** (Damped trend) | (A_d,N) Damped trend | (A_d,A) Damped HW add | (A_d,M) Damped HW mul |

### Full ETS models (with error type)

Adding the error component E = {A (additive), M (multiplicative)} doubles
the count to **18 models**:

$$
\text{ETS}(\underbrace{E}_{\text{Error}}, \underbrace{T}_{\text{Trend}}, \underbrace{S}_{\text{Seasonal}})
$$

- E = {A, M} (2 options)
- T = {N, A, A_d} (3 options)
- S = {N, A, M} (3 options)
- Total: 2 x 3 x 3 = **18 models**

However, **3 models are unstable** (multiplicative error with additive
components can produce negative forecasts), leaving **15 usable models**.

The unstable combinations are: ETS(M,N,A), ETS(M,A,A), ETS(M,A_d,A).

In [ ]:
# Visualize the ETS taxonomy
taxonomy = pd.DataFrame(
    index=["N (no trend)", "A (additive)", "Ad (damped)"],
    columns=["N (none)", "A (additive)", "M (multiplicative)"],
    data=[
        ["(N,N) SES", "(N,A)", "(N,M)"],
        ["(A,N) Holt", "(A,A) Add HW", "(A,M) Mul HW"],
        ["(Ad,N) Damped", "(Ad,A) Damp Add HW", "(Ad,M) Damp Mul HW"],
    ],
)
taxonomy.index.name = "Trend"
taxonomy.columns.name = "Seasonal"
print("ETS Method Taxonomy (9 methods, before adding error type):")
print()
print(taxonomy.to_string())

---
## 3. Additive vs Multiplicative Errors

The difference between additive and multiplicative errors lies in how
the observation relates to the state:

**Additive error**: $y_t = \mu_t + \varepsilon_t$ where $\varepsilon_t \sim N(0, \sigma^2)$

**Multiplicative error**: $y_t = \mu_t(1 + \varepsilon_t)$ where $\varepsilon_t \sim N(0, \sigma^2)$

where $\mu_t$ is the one-step-ahead forecast.

Key implications:
- Additive errors: constant variance of forecast errors.
- Multiplicative errors: variance of errors proportional to the forecast
  level (heteroscedastic).
- The **point forecasts** are identical for additive and multiplicative
  error versions of the same method (they differ only in prediction
  intervals and likelihood).

In [ ]:
# Compare ETS(A,A,A) vs ETS(M,A,A) point forecasts
# Note: ETS(M,A,A) is one of the unstable models, so we compare ETS(A,A,M) vs ETS(M,A,M)
ets_aam = ETSModel(
    train["Passengers"],
    error="add",
    trend="add",
    seasonal="mul",
    seasonal_periods=12,
).fit()

ets_mam = ETSModel(
    train["Passengers"],
    error="mul",
    trend="add",
    seasonal="mul",
    seasonal_periods=12,
).fit()

fc_aam = ets_aam.forecast(HORIZON)
fc_mam = ets_mam.forecast(HORIZON)

print(f"ETS(A,A,M) AIC: {ets_aam.aic:.2f}")
print(f"ETS(M,A,M) AIC: {ets_mam.aic:.2f}")
print(f"\nPoint forecasts differ because parameters are estimated")
print(f"under different likelihood functions.")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train["Passengers"].iloc[-36:], color="black", alpha=0.4, label="Train")
ax.plot(test["Passengers"], color="tab:blue", linewidth=2, label="Actual")
ax.plot(fc_aam, label="ETS(A,A,M)", color="tab:red", linestyle="--")
ax.plot(fc_mam, label="ETS(M,A,M)", color="tab:green", linestyle="--")
ax.set_title("Additive vs Multiplicative Error — Same Trend & Seasonal")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. statsmodels ETSModel

The `ETSModel` class in statsmodels provides the full state-space ETS
framework with proper maximum likelihood estimation.

In [ ]:
# Fit ETS(M,Ad,M) — multiplicative error, damped trend, multiplicative seasonal
ets_model = ETSModel(
    train["Passengers"],
    error="mul",
    trend="add",
    damped_trend=True,
    seasonal="mul",
    seasonal_periods=12,
)
ets_fit = ets_model.fit()

print(f"Model: ETS(M,Ad,M)")
print(f"Alpha:  {ets_fit.params['smoothing_level']:.4f}")
print(f"Beta*:  {ets_fit.params['smoothing_trend']:.4f}")
print(f"Gamma:  {ets_fit.params['smoothing_seasonal']:.4f}")
print(f"Phi:    {ets_fit.params['damping_trend']:.4f}")
print(f"AIC:    {ets_fit.aic:.2f}")
print(f"BIC:    {ets_fit.bic:.2f}")
print(f"AICc:   {ets_fit.aicc:.2f}")

In [ ]:
ets_fit.summary()

In [ ]:
fc_ets = ets_fit.forecast(HORIZON)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train["Passengers"], label="Train", color="black", alpha=0.5)
ax.plot(test["Passengers"], label="Actual (test)", color="tab:blue", linewidth=2)
ax.plot(fc_ets, label="ETS(M,Ad,M)", color="tab:red", linestyle="--", linewidth=2)
ax.set_title("ETS(M,Ad,M) Forecast — Airline Passengers")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

mae_ets = mean_absolute_error(test["Passengers"], fc_ets)
print(f"MAE: {mae_ets:.2f}")

---
## 5. Model Selection with AIC/BIC

The key advantage of the ETS framework is **automatic model selection**.
We fit all valid ETS models and choose the one with the lowest information
criterion.

- **AIC** (Akaike Information Criterion): balances fit and number of
  parameters. Lower is better.
- **BIC** (Bayesian Information Criterion): penalizes complexity more
  heavily than AIC. Lower is better.
- **AICc**: corrected AIC for small sample sizes. Preferred when
  $T/k < 40$ (T = sample size, k = number of parameters).

In [ ]:
def fit_all_ets(train_data, seasonal_periods=12):
    """Fit all valid ETS models and return results sorted by AICc."""
    error_types = ["add", "mul"]
    trend_types = [("add", False), ("add", True), (None, False)]  # A, Ad, N
    seasonal_types = ["add", "mul", None]

    # Unstable combinations to skip:
    # Multiplicative error with additive seasonal
    unstable = {
        ("mul", "add", False, "add"),   # ETS(M,A,A)
        ("mul", "add", True, "add"),    # ETS(M,Ad,A)
        ("mul", None, False, "add"),    # ETS(M,N,A)
    }

    results = []
    for error in error_types:
        for trend, damped in trend_types:
            for seasonal in seasonal_types:
                # Skip unstable combinations
                if (error, trend, damped, seasonal) in unstable:
                    continue

                # Build label
                e_label = "A" if error == "add" else "M"
                if trend is None:
                    t_label = "N"
                elif damped:
                    t_label = "Ad"
                else:
                    t_label = "A"
                s_label = {"add": "A", "mul": "M", None: "N"}[seasonal]
                label = f"ETS({e_label},{t_label},{s_label})"

                try:
                    sp = seasonal_periods if seasonal is not None else None
                    model = ETSModel(
                        train_data,
                        error=error,
                        trend=trend,
                        damped_trend=damped,
                        seasonal=seasonal,
                        seasonal_periods=sp,
                    )
                    fit = model.fit(disp=False)
                    results.append({
                        "Model": label,
                        "AIC": fit.aic,
                        "AICc": fit.aicc,
                        "BIC": fit.bic,
                        "fit": fit,
                    })
                except Exception:
                    pass  # Some combinations may fail to converge

    df = pd.DataFrame(results).sort_values("AICc").reset_index(drop=True)
    return df


print("fit_all_ets() defined — fits all valid ETS models.")

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ets_results = fit_all_ets(train["Passengers"], seasonal_periods=12)

# Display results (without the fit objects)
display_df = ets_results[["Model", "AIC", "AICc", "BIC"]].round(2)
print(f"Fitted {len(display_df)} ETS models:\n")
print(display_df.to_string(index=False))

In [ ]:
# Best model by AICc
best_model_name = ets_results.iloc[0]["Model"]
best_fit = ets_results.iloc[0]["fit"]

print(f"Best model by AICc: {best_model_name}")
print(f"AICc: {ets_results.iloc[0]['AICc']:.2f}")
print(f"\nSecond best: {ets_results.iloc[1]['Model']} (AICc: {ets_results.iloc[1]['AICc']:.2f})")
print(f"Third best:  {ets_results.iloc[2]['Model']} (AICc: {ets_results.iloc[2]['AICc']:.2f})")

In [ ]:
best_fit.summary()

---
## 6. AIC-Selected Model vs Manual Choice

In [ ]:
# Forecast from best AICc model
fc_best = best_fit.forecast(HORIZON)

# Manual choice: ETS(M,A,M) — multiplicative error, additive trend, multiplicative seasonal
manual_fit = ETSModel(
    train["Passengers"],
    error="mul",
    trend="add",
    seasonal="mul",
    seasonal_periods=12,
).fit(disp=False)
fc_manual = manual_fit.forecast(HORIZON)

actual = test["Passengers"].values

print(f"AIC-selected ({best_model_name}):")
print(f"  AICc: {best_fit.aicc:.2f}")
print(f"  MAE:  {mean_absolute_error(actual, fc_best):.2f}")
print(f"\nManual choice (ETS(M,A,M)):")
print(f"  AICc: {manual_fit.aicc:.2f}")
print(f"  MAE:  {mean_absolute_error(actual, fc_manual):.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train["Passengers"].iloc[-36:], color="black", alpha=0.4, label="Train")
ax.plot(test["Passengers"], color="tab:blue", linewidth=2, label="Actual")
ax.plot(fc_best, label=f"Best AICc: {best_model_name}",
        color="tab:red", linestyle="--", linewidth=2)
ax.plot(fc_manual, label="Manual: ETS(M,A,M)",
        color="tab:green", linestyle="--", linewidth=2)
ax.set_title("AIC-Selected vs Manual Model")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. AIC Comparison Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

models = display_df["Model"].values
aics = display_df["AICc"].values
colors = ["tab:red" if i == 0 else "tab:blue" for i in range(len(models))]

bars = ax.barh(range(len(models)), aics, color=colors, alpha=0.7)
ax.set_yticks(range(len(models)))
ax.set_yticklabels(models, fontsize=9)
ax.set_xlabel("AICc (lower is better)")
ax.set_title("AICc for All Valid ETS Models — Airline Passengers")
ax.invert_yaxis()

# Highlight best
ax.annotate(f"Best: {models[0]}", xy=(aics[0], 0),
            fontsize=10, fontweight="bold", color="tab:red",
            xytext=(20, 0), textcoords="offset points")

plt.tight_layout()
plt.show()

---
## 8. Prediction Intervals from ETS

In [ ]:
# Get prediction intervals from the best model
forecast_obj = best_fit.get_forecast(HORIZON)
forecast_df = forecast_obj.summary_frame(alpha=0.05)  # 95% confidence

print("Forecast summary (first 6 months):")
print(forecast_df.head(6).round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train["Passengers"].iloc[-48:], label="Train", color="black", alpha=0.5)
ax.plot(test["Passengers"], label="Actual (test)", color="tab:blue", linewidth=2)

# Point forecast
ax.plot(forecast_df["mean"], label=f"Forecast ({best_model_name})",
        color="tab:red", linewidth=2)

# 95% prediction intervals
ax.fill_between(
    forecast_df.index,
    forecast_df["pi_lower"],
    forecast_df["pi_upper"],
    color="tab:red", alpha=0.2, label="95% PI",
)

ax.set_title(f"{best_model_name} with 95% Prediction Intervals")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

# Check how many test points fall within the 95% PI
in_interval = (
    (test["Passengers"].values >= forecast_df["pi_lower"].values)
    & (test["Passengers"].values <= forecast_df["pi_upper"].values)
)
coverage = in_interval.mean() * 100
print(f"\nEmpirical coverage: {coverage:.1f}% of test points within 95% PI")

---
## 9. ETS Components Visualization

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Data
axes[0].plot(train["Passengers"], color="tab:blue")
axes[0].set_title(f"{best_model_name} — Data")
axes[0].set_ylabel("Passengers")

# Level
axes[1].plot(best_fit.level, color="tab:red")
axes[1].set_title("Level")
axes[1].set_ylabel("Level")

# Trend (slope)
axes[2].plot(best_fit.trend, color="tab:green")
axes[2].set_title("Trend")
axes[2].set_ylabel("Trend")

# Seasonal
axes[3].plot(best_fit.season, color="tab:purple")
axes[3].set_title("Seasonal")
axes[3].set_ylabel("Seasonal")

plt.tight_layout()
plt.show()

---
## 10. Fitted Values and Residuals

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Fitted values
axes[0].plot(train["Passengers"], label="Actual", color="tab:blue", alpha=0.7)
axes[0].plot(best_fit.fittedvalues, label="Fitted", color="tab:red",
             linestyle="--", alpha=0.8)
axes[0].set_title(f"{best_model_name} — Fitted Values")
axes[0].set_ylabel("Passengers")
axes[0].legend()

# Residuals
resid = best_fit.resid
axes[1].plot(resid, color="tab:blue", alpha=0.7)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set_title("Residuals")
axes[1].set_ylabel("Residual")

plt.tight_layout()
plt.show()

print(f"Mean residual: {resid.mean():.4f}")
print(f"Std residual:  {resid.std():.4f}")

---
## 11. ETSModel vs ExponentialSmoothing — When to Use Which

statsmodels provides **two** classes for exponential smoothing:

| Class | Module | Key feature |
|---|---|---|
| `ExponentialSmoothing` | `tsa.holtwinters` | Traditional Holt-Winters, simulation-based PIs |
| `ETSModel` | `tsa.exponential_smoothing.ets` | Full state-space ETS, analytical PIs, proper AIC |

**Use `ETSModel` when:**
- You need proper AIC/BIC for model selection.
- You want analytical prediction intervals.
- You are doing automatic model selection.

**Use `ExponentialSmoothing` when:**
- You want a quick Holt-Winters fit.
- You need multiplicative trend (not supported by `ETSModel`).
- You are manually specifying the model.

In [ ]:
# Compare the two implementations
# ExponentialSmoothing
hw_fit = ExponentialSmoothing(
    train["Passengers"],
    trend="add",
    seasonal="mul",
    seasonal_periods=12,
    initialization_method="estimated",
).fit(optimized=True)

# ETSModel equivalent: ETS(A,A,M)
ets_aam_fit = ETSModel(
    train["Passengers"],
    error="add",
    trend="add",
    seasonal="mul",
    seasonal_periods=12,
).fit(disp=False)

fc_hw = hw_fit.forecast(HORIZON)
fc_ets_aam = ets_aam_fit.forecast(HORIZON)

print("ExponentialSmoothing (HW):")
print(f"  Alpha: {hw_fit.params['smoothing_level']:.4f}")
print(f"  Beta:  {hw_fit.params['smoothing_trend']:.4f}")
print(f"  Gamma: {hw_fit.params['smoothing_seasonal']:.4f}")
print(f"  AIC:   {hw_fit.aic:.2f}")
print(f"  MAE:   {mean_absolute_error(test['Passengers'], fc_hw):.2f}")

print(f"\nETSModel ETS(A,A,M):")
print(f"  Alpha: {ets_aam_fit.params['smoothing_level']:.4f}")
print(f"  Beta:  {ets_aam_fit.params['smoothing_trend']:.4f}")
print(f"  Gamma: {ets_aam_fit.params['smoothing_seasonal']:.4f}")
print(f"  AIC:   {ets_aam_fit.aic:.2f}")
print(f"  MAE:   {mean_absolute_error(test['Passengers'], fc_ets_aam):.2f}")

---
## 12. Out-of-Sample Comparison: All Candidate Models

In [ ]:
# Compute out-of-sample MAE for all fitted ETS models
oos_results = []
for _, row in ets_results.iterrows():
    fit_obj = row["fit"]
    fc = fit_obj.forecast(HORIZON)
    mae = mean_absolute_error(test["Passengers"], fc)
    oos_results.append({
        "Model": row["Model"],
        "AICc": row["AICc"],
        "MAE (test)": mae,
    })

oos_df = pd.DataFrame(oos_results).round(2)
oos_df = oos_df.sort_values("MAE (test)").reset_index(drop=True)
print("All ETS models ranked by out-of-sample MAE:\n")
print(oos_df.to_string(index=False))

In [ ]:
# Does AICc agree with out-of-sample MAE?
aic_best = ets_results.iloc[0]["Model"]
mae_best = oos_df.iloc[0]["Model"]

print(f"Best by AICc:          {aic_best}")
print(f"Best by test-set MAE:  {mae_best}")

if aic_best == mae_best:
    print("\nAICc and MAE agree on the best model.")
else:
    print(f"\nThey disagree — AICc uses in-sample fit, MAE uses out-of-sample.")
    print("This is normal. AICc is an estimate of out-of-sample performance,")
    print("but it is not guaranteed to match the actual test-set ranking.")

---
## 13. Top Models Side by Side

In [ ]:
# Plot top 3 models by AICc
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train["Passengers"].iloc[-36:], color="black", alpha=0.4, label="Train")
ax.plot(test["Passengers"], color="tab:blue", linewidth=2, label="Actual")

colors = ["tab:red", "tab:green", "tab:purple"]
for i in range(min(3, len(ets_results))):
    row = ets_results.iloc[i]
    fc = row["fit"].forecast(HORIZON)
    mae = mean_absolute_error(test["Passengers"], fc)
    ax.plot(fc, color=colors[i], linestyle="--", linewidth=1.5,
            label=f"{row['Model']} (MAE={mae:.1f})")

ax.set_title("Top 3 ETS Models by AICc")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

---
## 14. When to Use Which ETS Variant

| Data characteristics | Recommended ETS | Rationale |
|---|---|---|
| No trend, no seasonality | ETS(A,N,N) or ETS(M,N,N) | SES equivalent |
| Trend, no seasonality | ETS(A,Ad,N) or ETS(M,Ad,N) | Damped trend usually best |
| No trend, additive seasonal | ETS(A,N,A) | Constant seasonal amplitude |
| No trend, multiplicative seasonal | ETS(A,N,M) or ETS(M,N,M) | Growing seasonal amplitude |
| Trend + additive seasonal | ETS(A,Ad,A) | Both components, damped |
| Trend + multiplicative seasonal | ETS(M,Ad,M) | Common for economic data |
| Not sure | **Fit all, pick lowest AICc** | Let the data decide |

**General guidelines:**
- **Always try damped trend** — it rarely hurts and often helps.
- **Multiplicative seasonality** is more common than additive in practice
  (most real-world seasonal effects scale with the level).
- **Multiplicative error** is appropriate when forecast errors are
  proportional to the forecast level.

In [ ]:
# Quick demonstration on data without seasonality
births = pd.read_csv(
    DATA_DIR / "DailyTotalFemaleBirths.csv",
    index_col="Date",
    parse_dates=True,
)
births.columns = ["Births"]
births_train = births.iloc[:335]
births_test = births.iloc[335:]

# Fit ETS(A,N,N) — SES equivalent
ets_ann = ETSModel(
    births_train["Births"],
    error="add",
    trend=None,
    seasonal=None,
).fit(disp=False)

fc_ann = ets_ann.forecast(len(births_test))
mae_ann = mean_absolute_error(births_test["Births"], fc_ann)

print(f"ETS(A,N,N) on daily births:")
print(f"  Alpha: {ets_ann.params['smoothing_level']:.4f}")
print(f"  AICc:  {ets_ann.aicc:.2f}")
print(f"  MAE:   {mae_ann:.2f}")
print(f"\nAs expected, ETS(A,N,N) = SES — appropriate for no-trend, no-seasonal data.")

---
## 15. The Complete Exponential Smoothing Hierarchy

Looking back across all four notebooks, we have built up the exponential
smoothing family step by step:

```
SES (1 parameter: alpha)
 |-> Holt's linear trend (2 parameters: alpha, beta*)
      |-> Holt's damped trend (3 parameters: alpha, beta*, phi)
           |-> Holt-Winters additive (3 parameters: alpha, beta*, gamma)
           |-> Holt-Winters multiplicative (3 parameters: alpha, beta*, gamma)
                |-> Damped HW (4 parameters: alpha, beta*, gamma, phi)
                     |-> Full ETS framework (15 usable models)
```

The ETS framework provides the **unified theory** that connects all
these methods and enables automatic model selection.

In [ ]:
# Final summary: parameters for the best airline model
print(f"Best airline model: {best_model_name}")
print(f"")
print(f"Parameters:")
for key, val in best_fit.params.items():
    if isinstance(val, (int, float, np.floating)):
        print(f"  {key}: {val:.4f}")
    elif isinstance(val, np.ndarray) and len(val) <= 12:
        print(f"  {key}: [{', '.join(f'{v:.3f}' for v in val)}]")

---
## Summary

| Concept | Detail |
|---------|--------|
| **ETS** | Error, Trend, Seasonal — a state-space formulation |
| **Models** | E={A,M} x T={N,A,Ad} x S={N,A,M} = 18, minus 3 unstable = **15** |
| **Error types** | Additive (constant variance) vs Multiplicative (proportional variance) |
| **Model selection** | Fit all valid models, choose lowest AIC/AICc/BIC |
| **AIC vs AICc** | AICc adds small-sample correction; preferred when T/k < 40 |
| **statsmodels** | `ETSModel` for full ETS; `ExponentialSmoothing` for quick HW |
| **Prediction intervals** | Analytical from state-space formulation |
| **Best practice** | Always try damped trend; let AICc choose among candidates |

**The ETS framework completes our coverage of exponential smoothing.  Next,
we move to ARIMA models — a fundamentally different approach based on
autocorrelation structure rather than weighted averages.**

In [ ]:
print("Chapter 07 complete.")
print("\nKey takeaway: the ETS framework provides a principled way to")
print("automatically select the best exponential smoothing model for")
print("any given time series, using information criteria (AIC/BIC).")